In [ ]:
import json
import os
import pandas as pd
from datetime import datetime, timezone

# --- Configuration ---
SOURCE_DIR = "./data/raw"
OUTPUT_DIR = "./data/json_clean"

# Fields to keep (from raw JSONL)
RAW_FIELDS = [
    'id','created_utc', 'author', 'author_fullname', 'subreddit',
    'subreddit_subscribers', 'url', 'ups', 'upvote_ratio',
    'num_comments', 'score', 'permalink', 'author_flair_text',
    'link_flair_text', 'selftext', 'title'
]

# Create output folder
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Collect all .jsonl files (including subfolders)
jsonl_files = []
for root, dirs, files in os.walk(SOURCE_DIR):
    # Skip the output folder itself
    if 'json_clean' in root:
        continue
    for f in sorted(files):
        if f.endswith('.jsonl'):
            jsonl_files.append(os.path.join(root, f))

print(f"Found {len(jsonl_files)} JSONL files\n")

skipped = []
processed = []
audit_rows = []

for filepath in jsonl_files:
    filename = os.path.basename(filepath)

    # Skip files not synced from OneDrive (start with null bytes)
    with open(filepath, 'rb') as fh:
        first_byte = fh.read(1)
    if first_byte == b'\x00' or first_byte == b'':
        skipped.append(filename)
        audit_rows.append({'file': filename, 'status': 'skipped', 'raw_lines': 0, 'cleaned_records': 0})
        continue

    # Process file
    cleaned_records = []
    raw_line_count = 0
    with open(filepath, 'r', encoding='utf-8') as fh:
        for line in fh:
            raw_line_count += 1
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            # Build cleaned record with desired fields
            cleaned = {}
            cleaned['reddit_post_id'] = record.get('id')
            for field in RAW_FIELDS:
                cleaned[field] = record.get(field)
            cleaned['source_file'] = filename

            # Derive datetime fields from created_utc
            utc_ts = record.get('created_utc')
            if utc_ts:
                try:
                    dt = datetime.fromtimestamp(float(utc_ts), tz=timezone.utc)
                    cleaned['created_dt'] = dt.strftime('%Y-%m-%d %H:%M:%S')
                    cleaned['created_date'] = dt.strftime('%Y-%m-%d')
                except (ValueError, OSError):
                    cleaned['created_dt'] = None
                    cleaned['created_date'] = None
            else:
                cleaned['created_dt'] = None
                cleaned['created_date'] = None

            cleaned_records.append(cleaned)

    # Save cleaned file as .jsonl
    out_path = os.path.join(OUTPUT_DIR, filename)
    with open(out_path, 'w', encoding='utf-8') as fh:
        for rec in cleaned_records:
            fh.write(json.dumps(rec) + '\n')

    processed.append((filename, len(cleaned_records)))
    audit_rows.append({
        'file': filename,
        'status': 'processed',
        'raw_lines': raw_line_count,
        'cleaned_records': len(cleaned_records)
    })
    print(f"✓ {filename} → {len(cleaned_records):,} records (raw lines: {raw_line_count:,})")

# Save audit log as CSV
audit_df = pd.DataFrame(audit_rows)
audit_csv = os.path.join(OUTPUT_DIR, 'step1_raw_to_clean_audit.csv')
audit_df.to_csv(audit_csv, index=False)

print(f"\n{'='*60}")
print(f"Processed: {len(processed)} files")
print(f"Skipped (not synced): {len(skipped)} files")
print(f"Total raw lines: {audit_df['raw_lines'].sum():,}")
print(f"Total cleaned records: {audit_df['cleaned_records'].sum():,}")
print(f"Audit log saved: {audit_csv}")
print(f"Output folder: {OUTPUT_DIR}")
if skipped:
    print(f"\nSkipped files:")
    for s in skipped:
        print(f"  ✗ {s}")
print(f"{'='*60}")
audit_df

### Remove duplicates

In [ ]:
import json
import os
import pandas as pd

CLEAN_DIR = "./data/json_clean"
DELETED_MARKERS = {'[deleted]', '[removed]', ''}

audit_rows = []

for filename in sorted(os.listdir(CLEAN_DIR)):
    if not filename.endswith('.jsonl'):
        continue

    filepath = os.path.join(CLEAN_DIR, filename)

    # Load all records
    records = []
    with open(filepath, 'r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    continue

    total_raw = len(records)

    # Step 1: Remove deleted/removed posts (selftext or title is [deleted]/[removed]/empty)
    records = [
        r for r in records
        if str(r.get('selftext', '')).strip() not in DELETED_MARKERS
        or str(r.get('title', '')).strip() not in DELETED_MARKERS
    ]
    after_deleted = len(records)

    # Step 2: Remove records where both title AND selftext are null/None
    records = [
        r for r in records
        if r.get('title') is not None or r.get('selftext') is not None
    ]
    after_nulls = len(records)

    # Step 3: Remove duplicates by reddit_post_id
    seen_ids = set()
    deduped = []
    for r in records:
        pid = r.get('reddit_post_id')
        if pid and pid not in seen_ids:
            seen_ids.add(pid)
            deduped.append(r)
        elif pid is None:
            deduped.append(r)
    records = deduped
    after_dedup = len(records)

    # Step 4: Create combined_text column from title + selftext
    for r in records:
        title = str(r.get('title', '') or '').strip()
        selftext = str(r.get('selftext', '') or '').strip()
        if title and selftext:
            r['combined_text'] = f"{title}. {selftext}"
        elif title:
            r['combined_text'] = title
        else:
            r['combined_text'] = selftext

    # Save cleaned file back
    with open(filepath, 'w', encoding='utf-8') as fh:
        for rec in records:
            fh.write(json.dumps(rec) + '\n')

    # Audit row
    audit_rows.append({
        'file': filename,
        'total_raw': total_raw,
        'removed_deleted': total_raw - after_deleted,
        'removed_nulls': after_deleted - after_nulls,
        'removed_duplicates': after_nulls - after_dedup,
        'total_removed': total_raw - after_dedup,
        'final_count': after_dedup
    })

    print(f"✓ {filename}: {total_raw:,} → {after_dedup:,} "
          f"(deleted:{total_raw - after_deleted}, nulls:{after_deleted - after_nulls}, "
          f"dupes:{after_nulls - after_dedup})")

# Save audit log as CSV
audit_df = pd.DataFrame(audit_rows)
audit_csv = os.path.join(CLEAN_DIR, 'cleaning_audit_log.csv')
audit_df.to_csv(audit_csv, index=False)

print(f"\n{'='*60}")
print(f"Files cleaned: {len(audit_rows)}")
print(f"Total records removed: {audit_df['total_removed'].sum():,}")
print(f"Total final records: {audit_df['final_count'].sum():,}")
print(f"Audit log saved: {audit_csv}")
print(f"{'='*60}")
audit_df

### Remove records where author field contain deleted

In [ ]:
import json
import os
import pandas as pd

CLEAN_DIR = "./data/json_clean"

audit_rows = []

for filename in sorted(os.listdir(CLEAN_DIR)):
    if not filename.endswith('.jsonl'):
        continue

    filepath = os.path.join(CLEAN_DIR, filename)

    # Load all records
    records = []
    with open(filepath, 'r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    continue

    total_before = len(records)

    # Step 1: Remove records where author contains '[deleted]'
    records = [r for r in records if '[deleted]' not in str(r.get('author', ''))]
    after_author = len(records)

    # Step 2: Remove records where selftext is empty string
    records = [r for r in records if r.get('selftext', '') != '']
    after_selftext = len(records)

    # Rebuild combined_text after filtering
    for r in records:
        title = str(r.get('title', '') or '').strip()
        selftext = str(r.get('selftext', '') or '').strip()
        if title and selftext:
            r['combined_text'] = f"{title}. {selftext}"
        elif title:
            r['combined_text'] = title
        else:
            r['combined_text'] = selftext

    # Save cleaned file back
    with open(filepath, 'w', encoding='utf-8') as fh:
        for rec in records:
            fh.write(json.dumps(rec) + '\n')

    audit_rows.append({
        'file': filename,
        'total_before': total_before,
        'removed_deleted_author': total_before - after_author,
        'removed_empty_selftext': after_author - after_selftext,
        'total_removed': total_before - after_selftext,
        'final_count': after_selftext
    })

    print(f"✓ {filename}: {total_before:,} → {after_selftext:,} "
          f"(deleted_author:{total_before - after_author:,}, empty_selftext:{after_author - after_selftext:,})")

# Save audit log
audit_df = pd.DataFrame(audit_rows)
audit_csv = os.path.join(CLEAN_DIR, 'cleaning_audit_step2.csv')
audit_df.to_csv(audit_csv, index=False)

print(f"\n{'='*60}")
print(f"Files cleaned: {len(audit_rows)}")
print(f"Total deleted authors removed: {audit_df['removed_deleted_author'].sum():,}")
print(f"Total empty selftext removed: {audit_df['removed_empty_selftext'].sum():,}")
print(f"Total records removed: {audit_df['total_removed'].sum():,}")
print(f"Total final records: {audit_df['final_count'].sum():,}")
print(f"Audit log saved: {audit_csv}")
print(f"{'='*60}")
audit_df

In [ ]:
import json
import os
import re
import pandas as pd

CLEAN_DIR = "./data/json_clean"

# --- Drug Name Normalization Mappings ---

# Brand name to RxNorm standard generic mapping
brand_to_generic = {
    # SSRIs
    'zoloft': 'sertraline', 'lustral': 'sertraline',
    'prozac': 'fluoxetine', 'sarafem': 'fluoxetine',
    'paxil': 'paroxetine', 'seroxat': 'paroxetine', 'brisdelle': 'paroxetine',
    'celexa': 'citalopram', 'cipramil': 'citalopram',
    'lexapro': 'escitalopram', 'cipralex': 'escitalopram',
    'luvox': 'fluvoxamine', 'faverin': 'fluvoxamine',
    # SNRIs
    'effexor': 'venlafaxine', 'effexor xr': 'venlafaxine',
    'cymbalta': 'duloxetine', 'irenka': 'duloxetine',
    'pristiq': 'desvenlafaxine', 'khedezla': 'desvenlafaxine',
    'savella': 'milnacipran',
    'fetzima': 'levomilnacipran',
    # Atypical
    'wellbutrin': 'bupropion', 'zyban': 'bupropion', 'aplenzin': 'bupropion',
    'remeron': 'mirtazapine', 'remeron soltab': 'mirtazapine',
    'desyrel': 'trazodone', 'oleptro': 'trazodone',
    'trintellix': 'vortioxetine', 'brintellix': 'vortioxetine',
    'viibryd': 'vilazodone',
    'serzone': 'nefazodone',
    # TCAs
    'elavil': 'amitriptyline', 'endep': 'amitriptyline',
    'pamelor': 'nortriptyline', 'aventyl': 'nortriptyline',
    'tofranil': 'imipramine',
    'sinequan': 'doxepin', 'silenor': 'doxepin',
    'norpramin': 'desipramine',
    'anafranil': 'clomipramine',
    'vivactil': 'protriptyline',
    'surmontil': 'trimipramine',
    'ludiomil': 'maprotiline',
    # MAOIs
    'nardil': 'phenelzine',
    'parnate': 'tranylcypromine',
    'marplan': 'isocarboxazid',
    'emsam': 'selegiline',
}

# Common misspellings to RxNorm standard
misspellings_to_generic = {
    'sertralina': 'sertraline', 'sertralin': 'sertraline', 'sertralline': 'sertraline',
    'fluoxetina': 'fluoxetine', 'fluoxetin': 'fluoxetine', 'fluxotine': 'fluoxetine',
    'paroxetina': 'paroxetine', 'paroxetin': 'paroxetine', 'paroxatine': 'paroxetine',
    'citalopran': 'citalopram', 'citalpram': 'citalopram',
    'escitalopran': 'escitalopram',
    'venlafaxina': 'venlafaxine', 'venlafaxin': 'venlafaxine', 'venlaflaxine': 'venlafaxine',
    'duloxetina': 'duloxetine', 'duloxetin': 'duloxetine',
    'buproprion': 'bupropion', 'bupropian': 'bupropion',
    'mirtazapin': 'mirtazapine', 'mirtazapina': 'mirtazapine', 'mirtazipine': 'mirtazapine',
    'amitriptiline': 'amitriptyline', 'amitriptilina': 'amitriptyline', 'amitriptaline': 'amitriptyline',
    'trazadone': 'trazodone', 'trazadon': 'trazodone',
}

# Combine all mappings, sort by length descending so multi-word matches go first
all_mappings = {**brand_to_generic, **misspellings_to_generic}
all_mappings_sorted = sorted(all_mappings.items(), key=lambda x: len(x[0]), reverse=True)

# Pre-compile regex patterns for performance
compiled_patterns = [(re.compile(r'\b' + re.escape(variant) + r'\b', re.IGNORECASE), generic)
                     for variant, generic in all_mappings_sorted]

print(f"Total normalization mappings: {len(all_mappings)}")

def normalize_drug_names(text):
    """Normalize antidepressant drug names to RxNorm standard generic names"""
    for pattern, generic in compiled_patterns:
        text = pattern.sub(generic, text)
    return text

# --- Process each file ---
audit_rows = []

for filename in sorted(os.listdir(CLEAN_DIR)):
    if not filename.endswith('.jsonl'):
        continue

    filepath = os.path.join(CLEAN_DIR, filename)

    records = []
    with open(filepath, 'r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    continue

    total_records = len(records)
    normalized_count = 0

    for r in records:
        original = r.get('combined_text', '')
        normalized = normalize_drug_names(str(original))
        if normalized != original:
            normalized_count += 1
        r['combined_text'] = normalized

    # Save back
    with open(filepath, 'w', encoding='utf-8') as fh:
        for rec in records:
            fh.write(json.dumps(rec) + '\n')

    audit_rows.append({
        'file': filename,
        'total_records': total_records,
        'records_normalized': normalized_count,
        'records_unchanged': total_records - normalized_count
    })

    print(f"✓ {filename}: {total_records:,} records, {normalized_count:,} normalized")

# Save audit log
audit_df = pd.DataFrame(audit_rows)
audit_csv = os.path.join(CLEAN_DIR, 'drug_normalization_audit.csv')
audit_df.to_csv(audit_csv, index=False)

print(f"\n{'='*60}")
print(f"Files processed: {len(audit_rows)}")
print(f"Total records: {audit_df['total_records'].sum():,}")
print(f"Total normalized: {audit_df['records_normalized'].sum():,}")
print(f"Audit log saved: {audit_csv}")
print(f"{'='*60}")
audit_df

### Keep records having antideprassant drug name(s)

In [ ]:
import json
import os
import re
import csv
import pandas as pd

CLEAN_DIR = "./data/json_clean"

# --- Build comprehensive antidepressant drug term set ---
# 1. From the drug list CSV
drug_terms = set()
with open('./data/1.Antidepressants_drug_list.csv', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    for row in reader:
        generic = row.get('Generic Name', '').strip()
        if generic:
            drug_terms.add(generic.lower())
        brands = row.get('Brand Name(s) (US/UK/Other)', '').strip()
        for b in brands.split(','):
            b = b.strip()
            # Remove parenthetical notes like "(Not used in UK)" or "(patch)"
            b = re.sub(r'\(.*?\)', '', b).strip()
            if b and len(b) > 2:
                drug_terms.add(b.lower())

# 2. Add all terms from the normalization mappings (brand names, misspellings, generics)
for variant, generic in all_mappings.items():
    drug_terms.add(variant.lower())
    drug_terms.add(generic.lower())

# Pre-compile word-boundary patterns for each drug term
drug_patterns = [(term, re.compile(r'\b' + re.escape(term) + r'\b', re.IGNORECASE))
                 for term in sorted(drug_terms, key=len, reverse=True)]

print(f"Total unique drug terms for matching: {len(drug_terms)}")
print(f"Sample terms: {sorted(list(drug_terms))[:10]}...")

def contains_antidepressant(text):
    """Check if text contains any antidepressant drug term"""
    text_lower = text.lower()
    for term, pattern in drug_patterns:
        if term in text_lower:  # fast pre-check
            if pattern.search(text):
                return True
    return False

# --- Process each file ---
audit_rows = []

for filename in sorted(os.listdir(CLEAN_DIR)):
    if not filename.endswith('.jsonl'):
        continue

    filepath = os.path.join(CLEAN_DIR, filename)

    records = []
    with open(filepath, 'r', encoding='utf-8') as fh:
        for line in fh:
            line = line.strip()
            if line:
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError:
                    continue

    total_before = len(records)

    # Keep only records where combined_text contains at least one antidepressant
    filtered = [r for r in records if contains_antidepressant(str(r.get('combined_text', '')))]
    total_after = len(filtered)

    # Save back
    with open(filepath, 'w', encoding='utf-8') as fh:
        for rec in filtered:
            fh.write(json.dumps(rec) + '\n')

    audit_rows.append({
        'file': filename,
        'total_before': total_before,
        'records_with_drug': total_after,
        'records_removed': total_before - total_after,
        'final_count': total_after
    })

    print(f"✓ {filename}: {total_before:,} → {total_after:,} (removed: {total_before - total_after:,})")

# Save audit log
audit_df = pd.DataFrame(audit_rows)
audit_csv = os.path.join(CLEAN_DIR, 'after_DrugNormalisation.csv')
audit_df.to_csv(audit_csv, index=False)

print(f"\n{'='*60}")
print(f"Files processed: {len(audit_rows)}")
print(f"Total before: {audit_df['total_before'].sum():,}")
print(f"Total with drug mention: {audit_df['records_with_drug'].sum():,}")
print(f"Total removed (no drug): {audit_df['records_removed'].sum():,}")
print(f"Audit log saved: {audit_csv}")
print(f"{'='*60}")
audit_df

### Create Batch Single Json file

In [ ]:
import json
import os
import pandas as pd

CLEAN_DIR = "./data/json_clean"
COMBINED_FILE = "./data/combined_all.jsonl"

audit_rows = []
total_combined = 0

with open(COMBINED_FILE, 'w', encoding='utf-8') as out_fh:
    for filename in sorted(os.listdir(CLEAN_DIR)):
        if not filename.endswith('.jsonl'):
            continue
        if filename.startswith('combined'):
            continue

        filepath = os.path.join(CLEAN_DIR, filename)
        count = 0

        with open(filepath, 'r', encoding='utf-8') as fh:
            for line in fh:
                line = line.strip()
                if line:
                    out_fh.write(line + '\n')
                    count += 1

        audit_rows.append({'file': filename, 'record_count': count})
        total_combined += count
        print(f"✓ {filename}: {count:,} records")

# Save audit log
audit_df = pd.DataFrame(audit_rows)
audit_csv = os.path.join(CLEAN_DIR, 'combined_jsonCount.csv')
audit_df.to_csv(audit_csv, index=False)

print(f"\n{'='*60}")
print(f"Files combined: {len(audit_rows)}")
print(f"Total records in combined file: {total_combined:,}")
print(f"Combined file: {COMBINED_FILE}")
print(f"Audit log: {audit_csv}")
print(f"{'='*60}")
audit_df

### Create Emoji Columns

In [ ]:
import json
import emoji

COMBINED_FILE = "./data/combined_all.jsonl"

# Load all records
records = []
with open(COMBINED_FILE, 'r', encoding='utf-8') as fh:
    for line in fh:
        line = line.strip()
        if line:
            records.append(json.loads(line))

emoji_count = 0

for r in records:
    text = str(r.get('combined_text', ''))
    # Extract emoji info: list of dicts with 'emoji' and 'match_start', etc.
    emoji_list = emoji.emoji_list(text)
    
    if emoji_list:
        r['is_emoji'] = 1
        # Get unique emoji names
        names = []
        seen = set()
        for e in emoji_list:
            ch = e['emoji']
            if ch not in seen:
                seen.add(ch)
                name = emoji.demojize(ch).strip(':').replace('_', ' ')
                names.append(name)
        r['emoji_name'] = ', '.join(names)
        emoji_count += 1
    else:
        r['is_emoji'] = 0
        r['emoji_name'] = ''

# Save back
with open(COMBINED_FILE, 'w', encoding='utf-8') as fh:
    for rec in records:
        fh.write(json.dumps(rec) + '\n')

print(f"Total records: {len(records):,}")
print(f"Records with emojis: {emoji_count:,}")
print(f"Records without emojis: {len(records) - emoji_count:,}")
print(f"Emoji prevalence: {emoji_count/len(records)*100:.2f}%")

### Final verification of duplicate records by reddit post id

In [ ]:
import json

COMBINED_FILE = "./data/combined_all.jsonl"

# Load all records
records = []
with open(COMBINED_FILE, 'r', encoding='utf-8') as fh:
    for line in fh:
        line = line.strip()
        if line:
            records.append(json.loads(line))

total_before = len(records)
print(f"Total records before dedup: {total_before:,}")

# Remove duplicates by reddit_post_id
seen_ids = set()
deduped = []
for r in records:
    pid = r.get('reddit_post_id')
    if pid and pid not in seen_ids:
        seen_ids.add(pid)
        deduped.append(r)
    elif pid is None:
        deduped.append(r)

total_after = len(deduped)
print(f"Total records after dedup:  {total_after:,}")
print(f"Duplicates removed:         {total_before - total_after:,}")

# Save back
with open(COMBINED_FILE, 'w', encoding='utf-8') as fh:
    for rec in deduped:
        fh.write(json.dumps(rec) + '\n')

print(f"\n✓ Deduplicated file saved: {COMBINED_FILE}")

### Calling LLM

In [ ]:
import os

# Set GEMINI_API_KEY in your environment before running (do not hardcode it here).
if not os.environ.get("GEMINI_API_KEY"):
    raise EnvironmentError(
        "GEMINI_API_KEY is not set. Export it in your shell or notebook environment before running."
    )


### Between 2015-01-01 and 2025-12-31

In [ ]:
import json

COMBINED_FILE = "./data/combined_all.jsonl"

# Read all records
with open(COMBINED_FILE, 'r', encoding='utf-8') as f:
    records = [json.loads(line) for line in f if line.strip()]

print(f"Before filtering: {len(records):,} records")

# Filter to 01.01.2015 – 31.12.2025
filtered = [r for r in records if '2015-01-01' <= r.get('created_date', '') <= '2025-12-31']

print(f"After filtering:  {len(filtered):,} records")

# Overwrite the file with filtered records
with open(COMBINED_FILE, 'w', encoding='utf-8') as f:
    for r in filtered:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

print(f"\n✅ combined_all.jsonl updated with {len(filtered):,} records (2015-01-01 to 2025-12-31)")

### Create Batch for LLM Extraction

In [ ]:
import json
import math
import pandas as pd
import os

COMBINED_FILE = "./data/combined_all.jsonl"
OUTPUT_DIR = "./data/LLM_results"
BATCH_MAP_FILE = os.path.join(OUTPUT_DIR, "batch_mapping.csv")
NUM_BATCHES = 50  # 494,665 / 50 ≈ 9,894 per batch (under 10,000 daily limit)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Load data and create batch mapping ──────────────────────────────────
print("Loading combined_all.jsonl...")
records = []
with open(COMBINED_FILE, 'r', encoding='utf-8') as fh:
    for line in fh:
        line = line.strip()
        if line:
            records.append(json.loads(line))

total = len(records)
batch_size = math.ceil(total / NUM_BATCHES)
print(f"Total records: {total:,}")
print(f"Number of batches: {NUM_BATCHES}")
print(f"Batch size: ~{batch_size:,} records per batch (under 10,000 daily limit)")

# Build and save batch mapping CSV (recreate to match filtered data)
if os.path.exists(BATCH_MAP_FILE):
    os.remove(BATCH_MAP_FILE)
    print("⚠ Removed old batch mapping (stale after date filtering)")

batch_rows = []
for i, rec in enumerate(records):
    batch_num = (i // batch_size) + 1
    batch_rows.append({
        'record_index': i,
        'reddit_post_id': rec.get('reddit_post_id', rec.get('id', '')),
        'batch_number': batch_num
    })
batch_df = pd.DataFrame(batch_rows)
batch_df.to_csv(BATCH_MAP_FILE, index=False)
print(f"\n✓ Batch mapping created and saved to: {BATCH_MAP_FILE}")

print(f"\nBatch summary:")
print(batch_df.groupby('batch_number').size().to_string())

# ── Show completion status for all batches ──────────────────────────────
print(f"\n{'='*60}")
print("Batch completion status:")
print(f"{'='*60}")
for b in range(1, NUM_BATCHES + 1):
    batch_file = os.path.join(OUTPUT_DIR, f"gemini_results_batch_{b}.csv")
    batch_total = len(batch_df[batch_df['batch_number'] == b])
    if os.path.exists(batch_file):
        done_df = pd.read_csv(batch_file)
        done_count = done_df['reddit_post_id'].nunique()
        status = "✓ COMPLETE" if done_count >= batch_total else f"⏳ {done_count:,}/{batch_total:,}"
    else:
        done_count = 0
        status = f"  NOT STARTED (0/{batch_total:,})"
    print(f"  Batch {b:2d}: {status}")
print(f"{'='*60}")

### LLM Extraction (Gemini 2.5 Flash Lite)
Change `BATCH_TO_RUN` below and re-run this cell for each batch. It auto-resumes from where it stopped.

In [ ]:
import json
import math
import pandas as pd
import asyncio
import os
import csv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import Optional
from tqdm.asyncio import tqdm as tqdm_asyncio
from tenacity import retry, wait_random_exponential, stop_after_attempt

COMBINED_FILE = "./data/combined_all.jsonl"
OUTPUT_DIR = "./data/LLM_results"
NUM_BATCHES = 50

# ════════════════════════════════════════════════════════════════════════
# ▶▶▶  SET THE BATCH NUMBER TO RUN HERE (1 to 50)  ◀◀◀
BATCH_TO_RUN = 14
# ════════════════════════════════════════════════════════════════════════

# ── Load data and compute batch slice ───────────────────────────────────
print(f"Loading combined_all.jsonl...")
records = []
with open(COMBINED_FILE, 'r', encoding='utf-8') as fh:
    for line in fh:
        line = line.strip()
        if line:
            records.append(json.loads(line))

total = len(records)
batch_size = math.ceil(total / NUM_BATCHES)

assert 1 <= BATCH_TO_RUN <= NUM_BATCHES, f"Batch must be between 1 and {NUM_BATCHES}"
start_idx = (BATCH_TO_RUN - 1) * batch_size
end_idx = min(BATCH_TO_RUN * batch_size, total)
batch_records = records[start_idx:end_idx]

# ── Check for existing partial results (resume support) ─────────────────
output_file = os.path.join(OUTPUT_DIR, f"gemini_results_batch_{BATCH_TO_RUN}.csv")
already_done_ids = set()
if os.path.exists(output_file):
    existing_df = pd.read_csv(output_file)
    already_done_ids = set(existing_df['reddit_post_id'].dropna().astype(str).tolist())
    print(f"⚠ Found existing results: {len(already_done_ids):,} unique records already processed")

remaining_indices = [
    i for i in range(len(batch_records))
    if str(batch_records[i].get('reddit_post_id', '')) not in already_done_ids
]
print(f"▶ Batch {BATCH_TO_RUN}: {len(batch_records):,} total, {len(remaining_indices):,} to process, {len(already_done_ids):,} already done")

if not remaining_indices:
    print(f"\n✓ Batch {BATCH_TO_RUN} is already complete! Nothing to do.")
    print(f"   {len(already_done_ids):,} records in: {output_file}")
else:
    # ── Define Pydantic schema ──────────────────────────────────────────
    class ExtractionResult(BaseModel):
        age: Optional[str] = Field(description="Age of the author if mentioned (e.g., '25', '30s', 'teenager'), otherwise null")
        gender: Optional[str] = Field(description="Gender or sex of the author if mentioned or implied (e.g., 'male', 'female', 'non-binary', 'M', 'F', '25M', '30F'), otherwise null")
        relevant_drugs: Optional[str] = Field(description="All brand names, generic names, or suspected medications mentioned, comma-separated")
        adverse_events: Optional[str] = Field(description="All symptoms, side effects, or medical events described, comma-separated")
        indication: Optional[str] = Field(description="The reason the patient was taking the medication (e.g., depression, anxiety, OCD)")
        outcome: Optional[str] = Field(description="Seriousness or outcome if mentioned (e.g., hospitalized, recovered, ongoing, discontinued)")
        causality_hints: Optional[str] = Field(description="Any mention of timing (e.g., '3 days after starting') or dechallenge/rechallenge (e.g., 'stopped after I quit the pill')")
        sentiment_perceptions: Optional[str] = Field(description="Emotional tone (e.g., anxious, frustrated, hopeful) and reasons for medication non-compliance or discontinuation if mentioned")
        drug_abuse_misuse: Optional[str] = Field(description="Any patterns of misuse, addiction, overdose, or slang/colloquialisms for illicit drug activities, otherwise null")
        off_label_use: Optional[str] = Field(description="Any mention of using the drug outside its approved indication, otherwise null")
        drug_drug_interaction: Optional[str] = Field(description="Any mention of multiple medications taken together that resulted in a reaction or interaction, otherwise null")
        location: Optional[str] = Field(description="Demographic location mentioned in the text (e.g., London, New York, Delhi, UK, USA, California), otherwise null")
        external_stressors: Optional[str] = Field(description="Major life events or environmental factors mentioned (e.g., COVID-19 infection, job loss, bereavement, financial stress, recent surgery, divorce, trauma), comma-separated, otherwise null")

    EXTRACTION_FIELDS = ['age', 'gender', 'relevant_drugs', 'adverse_events', 'indication', 'outcome',
                         'causality_hints', 'sentiment_perceptions', 'drug_abuse_misuse',
                         'off_label_use', 'drug_drug_interaction', 'location',
                         'external_stressors']

    # ── Initialize client ───────────────────────────────────────────────
    client = genai.Client()

    df_batch = pd.DataFrame(batch_records)
    all_columns = list(df_batch.columns) + EXTRACTION_FIELDS
    if not already_done_ids:
        with open(output_file, 'w', newline='', encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=all_columns)
            writer.writeheader()

    success_count = 0
    error_count = 0
    lock = asyncio.Lock()

    @retry(
        wait=wait_random_exponential(min=1, max=60),
        stop=stop_after_attempt(5),
        reraise=True
    )
    async def extract_and_save(text: str, local_idx: int, record: dict, semaphore: asyncio.Semaphore):
        global success_count, error_count
        async with semaphore:
            prompt = (
                "You are a pharmacovigilance expert. Extract the following information from this Reddit post about antidepressant medication. "
                "Be thorough and extract ALL relevant details. If information is not present, return null for that field.\n\n"
                f"Text:\n'''{text}'''"
            )

            try:
                response = await client.aio.models.generate_content(
                    model='gemini-2.5-flash-lite',
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json",
                        response_schema=ExtractionResult,
                        temperature=0.0
                    )
                )

                extracted = {}
                if response.parsed:
                    extracted = response.parsed.model_dump()

            except Exception as e:
                print(f"API Error on batch-index {local_idx}: {str(e)}")
                raise e

            row_out = dict(record)
            for field in EXTRACTION_FIELDS:
                row_out[field] = extracted.get(field)

            async with lock:
                with open(output_file, 'a', newline='', encoding='utf-8') as f:
                    writer = csv.DictWriter(f, fieldnames=all_columns)
                    writer.writerow({k: row_out.get(k) for k in all_columns})
                success_count += 1

    async def main():
        global success_count, error_count
        print(f"\nProcessing Batch {BATCH_TO_RUN}: {len(remaining_indices):,} records with Gemini 2.5 Flash Lite...")
        print(f"Results saving incrementally to: {output_file}\n")

        MAX_CONCURRENT_REQUESTS = 12
        semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

        tasks = []
        for local_idx in remaining_indices:
            rec = batch_records[local_idx]
            text = str(rec.get('combined_text', ''))
            tasks.append(extract_and_save(text, local_idx, rec, semaphore))

        for coro in tqdm_asyncio(asyncio.as_completed(tasks), total=len(tasks), desc=f"Batch {BATCH_TO_RUN}"):
            try:
                await coro
            except Exception as e:
                error_count += 1
                err_str = str(e)
                if any(k in err_str for k in ['RESOURCE_EXHAUSTED', '429', '503', 'UNAVAILABLE']):
                    print(f"\n⛔ Service unavailable or rate limit hit! {success_count:,} records saved so far.")
                    print(f"   Re-run this cell later to resume from where it stopped.")
                    break

        print(f"\n{'='*60}")
        print(f"Batch {BATCH_TO_RUN} Summary:")
        print(f"  ✓ Successfully processed: {success_count:,}")
        print(f"  ✗ Errors: {error_count:,}")
        print(f"  Results saved to: {output_file}")
        print(f"{'='*60}")

    await main()

### Validation on LLM extraction

In [ ]:
# Validation: Verify extracted_drug is present in combined_text
import pandas as pd

print("Loading results for validation...")
df_results = pd.read_csv('gemini_processed_results.csv')

# Function to check if drug is present in text
def validate_drug_in_text(row):
    """Check if extracted drug appears in the original text."""
    text = str(row['combined_text']).lower()
    drug = str(row['extracted_drug']).lower() if pd.notna(row['extracted_drug']) else None
    
    if drug is None or drug == 'nan' or drug == 'none':
        return {'present': False, 'reason': 'No drug extracted'}
    
    # Check exact match
    if drug in text:
        return {'present': True, 'reason': 'Exact match'}
    
    # Check partial matches (drug might be part of longer name)
    words = drug.split()
    if any(word in text for word in words if len(word) > 3):  # Check significant words
        return {'present': True, 'reason': 'Partial match'}
    
    return {'present': False, 'reason': 'Not found in text'}

# Apply validation
print("Validating extracted drugs...")
validation_results = df_results.apply(validate_drug_in_text, axis=1, result_type='expand')
df_results['drug_present'] = validation_results['present']
df_results['validation_reason'] = validation_results['reason']

# Statistics
total_rows = len(df_results)
total_extracted = df_results['extracted_drug'].notna().sum()
exact_matches = (validation_results['reason'] == 'Exact match').sum()
partial_matches = (validation_results['reason'] == 'Partial match').sum()
not_found = (validation_results['reason'] == 'Not found in text').sum()
no_extraction = (validation_results['reason'] == 'No drug extracted').sum()

print("\n" + "="*60)
print("VALIDATION RESULTS")
print("="*60)
print(f"Total records: {total_rows}")
print(f"Drugs extracted: {total_extracted} ({total_extracted/total_rows*100:.1f}%)")
print(f"\nValidation Breakdown:")
print(f"  ✓ Exact matches: {exact_matches} ({exact_matches/total_rows*100:.1f}%)")
print(f"  ✓ Partial matches: {partial_matches} ({partial_matches/total_rows*100:.1f}%)")
print(f"  ✗ Not found in text: {not_found} ({not_found/total_rows*100:.1f}%)")
print(f"  - No drug extracted: {no_extraction} ({no_extraction/total_rows*100:.1f}%)")
print(f"\nAccuracy (drugs present in text): {(exact_matches + partial_matches)/total_extracted*100:.1f}%")
print("="*60)

# Show examples of mismatches (drugs not found in text)
if not_found > 0:
    print(f"\n⚠️  Found {not_found} cases where extracted drug is NOT in text")
    print("\nSample mismatches:")
    mismatches = df_results[validation_results['reason'] == 'Not found in text'].head(5)
    for idx, row in mismatches.iterrows():
        print(f"\nRow {idx}:")
        print(f"  Extracted Drug: {row['extracted_drug']}")
        print(f"  Text Preview: {str(row['combined_text'])[:200]}...")
        print()

# Show validation summary by drug
if total_extracted > 0:
    print("\nValidation by Drug:")
    drug_validation = df_results[df_results['extracted_drug'].notna()].groupby('extracted_drug').agg({
        'drug_present': ['count', 'sum']
    })
    drug_validation.columns = ['Total', 'Present']
    drug_validation['Accuracy %'] = (drug_validation['Present'] / drug_validation['Total'] * 100).round(1)
    drug_validation = drug_validation.sort_values('Total', ascending=False).head(10)
    print(drug_validation)

# Save validation results
df_results.to_csv('gemini_processed_results_validated.csv', index=False)
print(f"\n✓ Validation results saved to 'gemini_processed_results_validated.csv'")